# 🧠 Custom CNN - Version Refactorisée

Ce notebook utilise les fonctions utilitaires de `src.notebooks.notebook_utils` pour un code plus propre et réutilisable.

## 1. Configuration et Imports

In [ ]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. Les variables sont prêtes à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, VOUS POUVEZ UTILISER:
- config: Objet de configuration (config.batch_size, config.data_dir, etc.)
- ENV: Environnement détecté ('colab', 'wsl', 'local')
- Tous les imports des transformers

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")
    
    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)
    
    os.chdir('/content/Data_Pipeline')
    
    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)
    
    # Installation du package en mode éditable (sans dépendances - détection Colab dans setup.py)
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")
    
    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])
    
    # Extraction models

    archive_models = '/content/drive/MyDrive/DS_COVID/inceptionv3_best.zip'
    if os.path.exists(archive_models):
        print("📦 Extraction models...")
        os.makedirs('./models/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_models, '-d', './models/'])



    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    # Depuis un notebook dans src/notebooks/
    project_root = Path.cwd().parent.parent

# Ajouter src/ au sys.path pour les imports
# src_path = str(project_root / 'src')
# if src_path not in sys.path:
#     sys.path.insert(0, src_path)
#     print(f"✅ Chemin src/ ajouté: {src_path}")

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

# Exports pour compatibilité avec anciens notebooks
data_dir = config.data_dir
categories = config.classes
img_size = config.img_size


# =============================================================================
# IMPORTS DES TRANSFORMERS
# =============================================================================

try:
    from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
    from src.features.Pipelines.Transformateurs.image_preprocessing import (
        ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker
    )
    from src.features.Pipelines.Transformateurs.image_augmentation import (
        ImageAugmenter, ImageRandomCropper
    )
    from src.features.Pipelines.Transformateurs.image_features import (
        ImageHistogram, ImagePCA, ImageStandardScaler
    )
    print("✅ Transformers importés")
except ImportError as e:
    print(f"⚠️ Erreur import transformers: {e}")


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB
# =============================================================================

plt.rcParams['figure.figsize'] = (15, 10)
sns.set_style('whitegrid')

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 70)
print("✅ CONFIGURATION PRÊTE - Data Pipeline")
print("=" * 70)
print(f"📂 Projet: {project_root}")
print(f"📊 Dataset: {data_dir}")
print(f"🏷️ Classes: {', '.join(categories)}")
print(f"🎛️ Images: {img_size}")
print(f"🔧 Batch: {config.batch_size} | Époques: {config.epochs}")
print(f"📐 Dataset accessible: {'✅' if data_dir.exists() else '❌'}")
print("=" * 70)
print("\n💡 Variables disponibles:")
print("   • config: Configuration complète (Config object)")
print("   • ENV: Environnement actuel")
print("\n🎯 Transformers disponibles:")
print("   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener")
print("   • ImageAugmenter, ImageRandomCropper")
print("   • ImageHistogram, ImagePCA, ImageStandardScaler")
print("=" * 70)
# Fin de la cellule de configuration standalone

In [ ]:
# Imports des fonctions utilitaires
from src.notebooks import (
    load_dataset,
    create_preprocessing_pipeline,
    prepare_train_val_test_split,
    compute_class_weights,
    create_data_generators,
    build_custom_cnn,
    compile_model,
    create_callbacks,
    train_model,
    evaluate_model,
    plot_training_curves,
    plot_confusion_matrix,
    select_sample_images,
    run_gradcam_analysis,
)

print("✅ Fonctions utilitaires importées")

## 2. Chargement et Préparation des Données

In [ ]:
# Charger le dataset
image_paths, mask_paths, labels, labels_int = load_dataset(
    data_dir=config.data_dir,
    categories=config.classes,
    n_images_per_class=None, # Charger toutes les images
    load_masks=False,  # True pour version maskée
    verbose=True
)

In [ ]:
# Créer la pipeline de preprocessing
pipeline = create_preprocessing_pipeline(
    img_size=(128, 128),
    color_mode='RGB',
    mask_paths=None,  # mask_paths pour version maskée
    verbose=True
)

# Charger et préprocesser les images
print("\nChargement des images...")
images = pipeline.fit_transform(image_paths)
images = images.astype('float32') / 255.0

print(f"\n📊 Images préparées:")
print(f"  Shape: {images.shape}")
print(f"  Range: [{images.min():.3f}, {images.max():.3f}]")
print(f"  Dtype: {images.dtype}")

In [ ]:
# Split train/val/test
X_train, X_val, X_test, y_train_cat, y_val_cat, y_test_cat = prepare_train_val_test_split(
    images=images,
    labels_int=labels_int,
    num_classes=len(config.classes),
    test_size=0.15,
    val_size=0.15,
    random_seed=config.random_seed,
    verbose=True
)

# Récupérer les labels integer pour le calcul des class weights
y_train = np.argmax(y_train_cat, axis=1)
y_val = np.argmax(y_val_cat, axis=1)
y_test = np.argmax(y_test_cat, axis=1)

In [ ]:
# Calculer les class weights
class_weights = compute_class_weights(
    y_train=y_train,
    categories=config.classes,
    verbose=True
)

In [ ]:
# Créer les data generators
config.batch_size = 512  # Ajuster si nécessaire

train_generator, val_generator = create_data_generators(
    X_train=X_train,
    y_train_cat=y_train_cat,
    X_val=X_val,
    y_val_cat=y_val_cat,
    batch_size=config.batch_size,
    augment_train=True,
    verbose=True
)

## 3. Construction et Compilation du Modèle

In [ ]:
# Construire le modèle
model = build_custom_cnn(
    input_shape=(128, 128, 3),
    num_classes=len(config.classes),
    verbose=True
)

# Afficher le résumé
model.summary()

In [ ]:
# Compiler le modèle
model = compile_model(
    model=model,
    learning_rate=0.001,
    verbose=True
)

In [ ]:
# Créer les callbacks
callbacks = create_callbacks(
    models_dir=config.results_dir / 'custom_cnn_models',
    monitor='val_accuracy',
    patience_early_stop=15,
    patience_reduce_lr=5,
    verbose=True
)

## 4. Entraînement

In [ ]:
# Entraîner le modèle
EPOCHS = 50

history = train_model(
    model=model,
    train_generator=train_generator,
    val_generator=val_generator,
    class_weights=class_weights,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=True
)

## 5. Visualisation des Courbes d'Apprentissage

In [ ]:
# Plot training curves
plots_dir = config.results_dir / 'custom_cnn_plots'
plots_dir.mkdir(parents=True, exist_ok=True)

fig = plot_training_curves(
    history=history,
    save_path=plots_dir / 'training_curves.png',
    figsize=(15, 12)
)
plt.show()

## 6. Évaluation

In [ ]:
# Évaluer le modèle
y_pred, y_pred_proba = evaluate_model(
    model=model,
    X_test=X_test,
    y_test_cat=y_test_cat,
    y_test=y_test,
    categories=config.classes,
    verbose=True
)

In [ ]:
# Matrice de confusion
fig = plot_confusion_matrix(
    y_test=y_test,
    y_pred=y_pred,
    categories=config.classes,
    save_path=plots_dir / 'confusion_matrix.png',
    figsize=(10, 8)
)
plt.show()

## 7. Interprétabilité - Grad-CAM

In [ ]:
# Sélectionner des images échantillons
sample_indices = select_sample_images(
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba,
    categories=config.classes,
    n_samples=6,
    random_seed=42,
    verbose=True
)

In [ ]:
# Analyse Grad-CAM
interp_dir = config.results_dir / 'interpretability_custom_cnn'
interp_dir.mkdir(parents=True, exist_ok=True)

gradcam, heatmaps = run_gradcam_analysis(
    model=model,
    X_test=X_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba,
    categories=config.classes,
    sample_indices=sample_indices,
    save_dir=interp_dir,
    verbose=True
)

plt.show()

## 8. Résumé

✅ Notebook refactorisé utilisant les fonctions utilitaires !

**Avantages:**
- Code plus court et lisible
- Fonctions réutilisables entre notebooks
- Facile à maintenir et débugger
- Paramètres centralisés

In [ ]:
print("=" * 70)
print("🎉 NOTEBOOK TERMINÉ")
print("=" * 70)
print("\n✅ Modèle entraîné et évalué")
print("✅ Visualisations générées")
print("✅ Interprétabilité analysée")